# OCSF-RainNet: GPU resource/latency profiling
Computational graph benchmarking for six variants. Five-class heads are **untrained**: these timings and parameter counts are not segmentation accuracy results. Same batch/shape and one GPU; synchronous timing.


In [ ]:
import os,sys,json,time,csv,datetime,gc,hashlib,platform
from pathlib import Path
import numpy as np
os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
import tensorflow as tf
os.environ["TF_CPP_MIN_LOG_LEVEL"]="2"
np.random.seed(42)
tf.random.set_seed(42)
gpu=tf.config.list_physical_devices("GPU")
if gpu:
    tf.config.set_visible_devices(gpu[:1],"GPU")
    tf.config.experimental.set_memory_growth(gpu[0],True)
INPUT=next((p.parent for p in Path("/kaggle/input").rglob("pilot_audit_summary.json")),None)
assert INPUT and INPUT.is_dir(),"Private Kaggle pilot dataset missing"
sys.path.insert(0,str(INPUT))
import rainnet_intensity_models as models
models.CHECKPOINT=INPUT/"rainnet_binary_best_05.h5"
models.ARCH=INPUT
assert models.pretrained_sha256()=="71e0e6d7e0fa9fa6f541c89f1bfcc57d701bd94730e0508c2dbf45c341de575f"
OUT=Path("/kaggle/working")
REPORT={"protocol":"UNTRAINED_GRAPH_INFERENCE_PROFILING_NOT_SEGMENTATION_SKILL","started_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),"tensorflow":tf.__version__,"gpu":gpu[0].name if gpu else "NO_GPU","hardware":platform.platform(),"warmups":6,"timed_iterations":30,"batch_size":1,"input_shapes":[[128,128,3],[192,464,3]],"is_paper_accuracy":False,"model_comparison_protocol":"Matched input size, batch, synchronous device timing, inference mode; no learned five-class weights"}
print("TF",tf.__version__,"GPU",gpu[0].name if gpu else "NO_GPU")
assert gpu,"GPU not enabled for Kaggle run"



In [ ]:
def benchmark(m,x,iterations=30,warmups=6):
    inp=tf.convert_to_tensor(x,dtype=tf.float32)
    for _ in range(warmups): _=m(inp,training=False).numpy()
    try:tf.config.experimental.reset_memory_stats("GPU:0")
    except Exception:pass
    timings=[]
    for _ in range(iterations):
        ts=time.perf_counter()
        p=m(inp,training=False).numpy()
        timings.append(1000*(time.perf_counter()-ts))
    assert p.shape[0]==1 and p.shape[-1]==5 and np.isfinite(p).all()
    max_prob_err=float(np.max(np.abs(p.sum(axis=-1)-1)))
    assert max_prob_err<1e-4
    peak=None
    try:peak=round(tf.config.experimental.get_memory_info("GPU:0")["peak"]/1024**2,3)
    except Exception:pass
    return {"median_ms":float(np.median(timings)),"p95_ms":float(np.percentile(timings,95)),"mean_ms":float(np.mean(timings)),"sd_ms":float(np.std(timings,ddof=1)),"peak_gpu_memory_mib":peak,"normalization_error_max":max_prob_err}
specs=[("RainNet-softmax","plain",142637),("RainNet-fusion","fusion",148234),("RainNet-ordinal","ordinal",142632),("OCSF-RainNet","proposed",148217),("U-Net","unet",485853),("RA-UNet-style","raunet",508249)]
rows=[]
for label,key,n_params in specs:
    tf.keras.backend.clear_session();gc.collect()
    try:
        model=models.build_unet_baseline(residual_attention=(key=="raunet")) if key in ("unet","raunet") else models.build_rainnet_variant(key,pretrained=True,freeze_backbone=True)
        assert model.count_params()==n_params,(label,model.count_params(),n_params)
        for H,W in [(128,128),(192,464)]:
            z=np.random.default_rng(2026).uniform(0,1,(1,H,W,3)).astype("float32")
            run=benchmark(model,z,iterations=30,warmups=6)
            row={"model":label,"parameters":n_params,"batch":1,"height":H,"width":W,"status":"PASS",**run}
            rows.append(row);print(json.dumps(row,sort_keys=True),flush=True)
    except Exception as exc:
        rows.append({"model":label,"status":"FAIL","error":repr(exc)})
        print("PROFILE_FAILED",label,repr(exc),flush=True)
REPORT["runs"]=rows
REPORT["finished_utc"]=datetime.datetime.now(datetime.timezone.utc).isoformat()
REPORT["verified_runs"]=sum(r["status"]=="PASS" for r in rows)
(OUT/"OCSF_RainNet_GPU_Profiling.json").write_text(json.dumps(REPORT,indent=2),encoding="utf8")
with open(OUT/"OCSF_RainNet_GPU_Profiling.csv","w",newline="",encoding="utf8") as f:
    keys=["model","parameters","batch","height","width","status","median_ms","p95_ms","mean_ms","sd_ms","peak_gpu_memory_mib","normalization_error_max","error"]
    w=csv.DictWriter(f,fieldnames=keys);w.writeheader()
    for row in rows:w.writerow(row)
assert REPORT["verified_runs"]==12, f"Some profile runs failed: {rows}"
print("PROFILING_COMPLETED",REPORT["verified_runs"],"NO TRAINING OR HOLDOUT ACCURACY CLAIMS")



In [ ]:
import matplotlib.pyplot as plt
labels=[a[0] for a in specs]
fig,ax=plt.subplots(figsize=(11,5))
data=[next(row["median_ms"] for row in rows if row["model"]==label and row.get("height")==192) for label in labels]
bars=ax.bar(labels,data)
ax.set_ylabel("Median synchronized inference time (ms)")
ax.set_title("OCSF-RainNet and matched comparators: GPU graph inference timing (untrained heads)")
ax.tick_params(axis="x",rotation=20)
for bar,v in zip(bars,data):ax.annotate(f"{v:.1f}",(bar.get_x()+bar.get_width()/2,v),xytext=(0,3),textcoords="offset points",ha="center",fontsize=9)
fig.tight_layout()
fig.savefig(OUT/"OCSF_RainNet_GPU_Inference_Latency_600dpi.png",dpi=600,bbox_inches="tight")
plt.close(fig)
print("OUTPUT_FILES",[(x.name,x.stat().st_size) for x in OUT.glob("OCSF_RainNet_*")])

